# A1.10 · Rogue agents in a multi-agent system

**Function A — Securing AI Architectures → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.9 · Agent communication poisoning](https://spbreed.github.io/cyber-commons/lessons/A1.9.html)**.

| | |
|---|---|
| Open-source tooling | SPIFFE/SPIRE, kagent |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The orchestrator delegates to whatever agents it discovers. Something joined the pool this morning that nobody registered, and it has been receiving work ever since, with the same standing as the agents you wrote.

## 2 · The framework

```
   registered           discovered at runtime
   +---------+          +---------+   +---------+
   | agent A |          | agent B |   |   ???   |  <- joined this morning
   +----+----+          +----+----+   +----+----+
        |                    |             |
        +-------- orchestrator delegates ---+

   the pool is a trust boundary. most orchestrators treat it as a config file.
```

**OWASP T13 — Rogue Agents in Multi-Agent Systems.**

The **orchestrator** delegates work to agents. The question this risk asks is
disarmingly simple: *how does it know which agents are allowed to receive that
work?*

In most deployments the answer is configuration — a list in a file, an env var,
a service discovery lookup. None of those is an identity check. An agent that
appears in the right place, answering the right protocol, is treated as a
legitimate worker.

Two ways one arrives:

**A compromised legitimate agent.** It was registered and approved; it is now
executing someone else's instructions after A1.3 or A1.9. Nothing about its
registration is wrong, which is why registration alone does not solve this.

**An unregistered agent.** A developer stood one up to test something, or an
attacker with a foothold registered a service. It receives delegated work and
delegated authority because the topology admits by convention rather than by
identity.

The consequence specific to multi-agent systems: **delegated authority flows to
it.** The orchestrator does not just send a task, it sends the context and often
a token. So an agent nobody approved ends up holding a credential that narrows
from a real user's, and the audit trail — if it records agent names at all —
records a name the attacker chose.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

An orchestrator that admits workers by configuration.

In [ ]:
REGISTRY = {"pricing-agent":  {"owner": "payments-team", "approved": True},
            "billing-agent":  {"owner": "payments-team", "approved": True}}

DISCOVERED = ["pricing-agent", "billing-agent", "reporting-agent-v2"]

DELEGATED = []

def delegate(agent_name, task, user_token):
    """The orchestrator hands work - and the caller's narrowed token - onward."""
    DELEGATED.append({"agent": agent_name, "task": task, "token": user_token})
    return f"{agent_name} accepted"

USER_TOKEN = "obo:dana@corp:reports:read,reports:write"

print(f"{'agent':22s}{'in registry?':14s}{'approved?':11s}received work?")
for name in DISCOVERED:
    entry = REGISTRY.get(name)
    delegate(name, "summarise Q3 revenue", USER_TOKEN)      # admitted by discovery
    print(f"{name:22s}{str(bool(entry)):14s}"
          f"{str(bool(entry and entry['approved'])):11s}yes")

rogue = [d for d in DELEGATED if d["agent"] not in REGISTRY]
print(f"\nagents that received delegated work : {len(DELEGATED)}")
print(f"of which unregistered               : {len(rogue)}")
for r in rogue:
    print(f"   {r['agent']} now holds {r['token']}")
print()
print("It was admitted because it answered the protocol in the right place.")
print("It received the task AND the narrowed user token, so it can act as dana")
print("against every downstream that honours that token.")
assert rogue and all(r["token"] == USER_TOKEN for r in rogue)

## What you just proved

Three agents are discovered, two are in the registry, and all three receive delegated work — including the narrowed user token. The unregistered agent can now act as the requesting user against any downstream that honours it.

## Your turn

Ask how your orchestrator decides which agents may receive work. If the answer is a config list or service discovery, write down what would have to be true for an extra entry to be noticed.

---

**Next → [A1.11 · Cascading hallucination](https://spbreed.github.io/cyber-commons/lessons/A1.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*